In [1]:
import os
import json
import sys
sys.path.append(os.path.abspath('../'))

from retrieval_models import *

with open('../data/data.json') as f:
    data = json.loads(f.read())

In [2]:
queries = data['queries_stopped_stemmed']
documents = data['documents_stopped_stemmed']
qrels = data['qrels']

print("Number of queries:", len(queries))
print("Number of documents,", len(documents))

Number of queries: 10
Number of documents, 229


In [3]:
def mean_per_key(scores: dict[str, dict[str, float]]) -> dict[str, float]:
    q_ids = [key for key in scores.keys()]
    key_lists = {k: [v] for k, v in scores[q_ids[0]].items()}
    for q_id in q_ids[1:]:
        q_scores = scores[q_id]
        for k, v in q_scores.items():
            key_lists[k].append(v)
    return {k: sum(v) / len(v) for k, v in key_lists.items()}

In [4]:
tf_results = compute_tf(queries, documents)
tf_scores = tf_results.compute_metrics(qrels)
mean_per_key(tf_scores)

{'map': 0.41563012421546974,
 'P_10': 0.42000000000000004,
 'recall_10': 0.71,
 'ndcg_cut_10': 0.43664253565172506}

In [5]:
bm25_results = compute_bm25(queries, documents)
bm25_scores = bm25_results.compute_metrics(qrels)
mean_per_key(bm25_scores)

{'map': 0.4453892473387232,
 'P_10': 0.43,
 'recall_10': 0.7333333333333333,
 'ndcg_cut_10': 0.47487553434605756}

In [6]:
ql_results = compute_ql(queries, documents)
ql_scores = ql_results.compute_metrics(qrels)
mean_per_key(ql_scores)

{'map': 0.4330957361113885,
 'P_10': 0.43,
 'recall_10': 0.7333333333333333,
 'ndcg_cut_10': 0.4927198609894753}

In [7]:
def examine_results(results: RetrievalModelScores, q_id: str, top_k: int):
    query_text = queries[q_id]
    print("=" * 20)
    print(q_id)
    print(" ".join(query_text))
    print("=" * 20)
    
    doc_scores = results.doc_scores[q_id]
    score_pairs = sorted([(d_id, doc.score) for d_id, doc in doc_scores.items()], key=lambda x: x[1], reverse=True)[:top_k]
    for d_id, score in score_pairs:
        print(d_id, score)
        print(" ".join(documents[d_id]))
        scored_doc = doc_scores[d_id]
        print("Word scores:")
        for word, score in scored_doc.word_scores.items():
            print('\t', word, score)
        if scored_doc.missing_word_scores:
            print("Missing word scores:")
            for word, score in scored_doc.missing_word_scores.items():
                print(word, score, end=" ")
        print()
        print()

In [8]:
examine_results(tf_results, '1110199', 10)

1110199
wifi vs bluetooth
398442 8
Relat : wifi repeat wifi router wifi long rang antenna wifi rang extend wireless router bluetooth transmitt wifi extend outdoor wifi antenna wifi antenna .
Word scores:
	 Relat 0
	 : 0
	 wifi 7
	 repeat 0
	 router 0
	 long 0
	 rang 0
	 antenna 0
	 extend 0
	 wireless 0
	 bluetooth 1
	 transmitt 0
	 outdoor 0
	 . 0


554521 6
New Arrival . kind Tablet Pc Wifi Bluetooth Camera avail LightInTheBox . guarante cool Tablet Pc Wifi Bluetooth Camera high qualiti afford price . Check websit buy favorit Tablet Pc Wifi Bluetooth Camera . Also , kind awesom product avail LightInTheBox .
Word scores:
	 New 0
	 Arrival 0
	 . 0
	 kind 0
	 Tablet 0
	 Pc 0
	 Wifi 0
	 Bluetooth 0
	 Camera 0
	 avail 0
	 LightInTheBox 0
	 guarante 0
	 cool 0
	 high 0
	 qualiti 0
	 afford 0
	 price 0
	 Check 0
	 websit 0
	 buy 0
	 favorit 0
	 Also 0
	 , 0
	 awesom 0
	 product 0
	 bluetooth 3
	 wifi 3


8160527 5
Bluetooth 4.0 vs. Wi-Fi Direct : Speed Wi-Fi Direct promis device-to-devic tr

In [9]:
examine_results(bm25_results, '1110199', 10)

1110199
wifi vs bluetooth
398442 9.915324354459125
Relat : wifi repeat wifi router wifi long rang antenna wifi rang extend wireless router bluetooth transmitt wifi extend outdoor wifi antenna wifi antenna .
Word scores:
	 Relat 0
	 : 0
	 wifi 6.575486499785576
	 repeat 0
	 router 0
	 long 0
	 rang 0
	 antenna 0
	 extend 0
	 wireless 0
	 bluetooth 3.339837854673549
	 transmitt 0
	 outdoor 0
	 . 0


554521 9.720344089212906
New Arrival . kind Tablet Pc Wifi Bluetooth Camera avail LightInTheBox . guarante cool Tablet Pc Wifi Bluetooth Camera high qualiti afford price . Check websit buy favorit Tablet Pc Wifi Bluetooth Camera . Also , kind awesom product avail LightInTheBox .
Word scores:
	 New 0
	 Arrival 0
	 . 0
	 kind 0
	 Tablet 0
	 Pc 0
	 Wifi 0
	 Bluetooth 0
	 Camera 0
	 avail 0
	 LightInTheBox 0
	 guarante 0
	 cool 0
	 high 0
	 qualiti 0
	 afford 0
	 price 0
	 Check 0
	 websit 0
	 buy 0
	 favorit 0
	 Also 0
	 , 0
	 awesom 0
	 product 0
	 bluetooth 4.518292987905949
	 wifi 5.202051101

In [10]:
examine_results(ql_results, '1110199', 10)

1110199
wifi vs bluetooth
8160527 -13.677129717154898
Bluetooth 4.0 vs. Wi-Fi Direct : Speed Wi-Fi Direct promis device-to-devic transfer speed 250Mbps , Bluetooth 4.0 promis speed similar Bluetooth 3.0 25Mbps . Bluetooth 4.0 Wi-Fi Direct use 802.11 network standard reach maximum speed .
Word scores:
	 Bluetooth 0
	 4.0 0
	 vs. 0
	 Wi-Fi 0
	 Direct 0
	 : 0
	 Speed 0
	 promis 0
	 device-to-devic 0
	 transfer 0
	 speed 0
	 250Mbps 0
	 , 0
	 similar 0
	 3.0 0
	 25Mbps 0
	 . 0
	 use 0
	 802.11 0
	 network 0
	 standard 0
	 reach 0
	 maximum 0
	 bluetooth -2.357196562898399
	 vs -3.7460272600929394
Missing word scores:
wifi -7.573905894163561 

398442 -14.162473238276611
Relat : wifi repeat wifi router wifi long rang antenna wifi rang extend wireless router bluetooth transmitt wifi extend outdoor wifi antenna wifi antenna .
Word scores:
	 Relat 0
	 : 0
	 wifi -1.4106200746987119
	 repeat 0
	 router 0
	 long 0
	 rang 0
	 antenna 0
	 extend 0
	 wireless 0
	 bluetooth -3.342449605945496
	 trans

In [11]:
# compute ranges of values
k_1_range = np.linspace(0.0, 2.0, 5).tolist() + [10.0]
b_range = np.linspace(0.0, 1.0, 6).tolist()
bm25_range = compute_bm25_range(queries, documents, k_1_range, b_range)
bm25_range[0.0][0.0].doc_scores

{'1110199': {'1729': <retrieval_models.ScoredDocument at 0x746b9e970ec0>,
  '47210': <retrieval_models.ScoredDocument at 0x746b9e971010>,
  '87404': <retrieval_models.ScoredDocument at 0x746b9e970d70>,
  '122857': <retrieval_models.ScoredDocument at 0x746b9e970980>,
  '172170': <retrieval_models.ScoredDocument at 0x746b9e9710f0>,
  '211625': <retrieval_models.ScoredDocument at 0x746b9e971160>,
  '266552': <retrieval_models.ScoredDocument at 0x746b9e9711d0>,
  '331936': <retrieval_models.ScoredDocument at 0x746b9e971240>,
  '370387': <retrieval_models.ScoredDocument at 0x746b9e9712b0>,
  '398442': <retrieval_models.ScoredDocument at 0x746b9e971320>,
  '418265': <retrieval_models.ScoredDocument at 0x746b9e971390>,
  '441197': <retrieval_models.ScoredDocument at 0x746b9e971400>,
  '444380': <retrieval_models.ScoredDocument at 0x746b9e971470>,
  '527690': <retrieval_models.ScoredDocument at 0x746b9e9714e0>,
  '554521': <retrieval_models.ScoredDocument at 0x746b9e971550>,
  '557605': <retri

In [14]:
lambda_range = np.linspace(0.0, 1.0, 6).tolist()
ql_range = compute_ql_range(queries, documents, lambda_range)
ql_range[0.0].doc_scores

{'1110199': {'1729': <retrieval_models.ScoredDocument at 0x746b96a7e660>,
  '47210': <retrieval_models.ScoredDocument at 0x746b96a7e6d0>,
  '87404': <retrieval_models.ScoredDocument at 0x746b96a7e890>,
  '122857': <retrieval_models.ScoredDocument at 0x746b96a7e9e0>,
  '172170': <retrieval_models.ScoredDocument at 0x746b96a7e5f0>,
  '211625': <retrieval_models.ScoredDocument at 0x746b96a7e580>,
  '266552': <retrieval_models.ScoredDocument at 0x746b96a7e510>,
  '331936': <retrieval_models.ScoredDocument at 0x746b96a7e4a0>,
  '370387': <retrieval_models.ScoredDocument at 0x746b96a7e430>,
  '398442': <retrieval_models.ScoredDocument at 0x746b96a7e3c0>,
  '418265': <retrieval_models.ScoredDocument at 0x746b96a7e350>,
  '441197': <retrieval_models.ScoredDocument at 0x746b96a7e2e0>,
  '444380': <retrieval_models.ScoredDocument at 0x746b96a7e200>,
  '527690': <retrieval_models.ScoredDocument at 0x746b96a7e270>,
  '554521': <retrieval_models.ScoredDocument at 0x746b96a7e190>,
  '557605': <retri